In [2]:
import pandas as pd
from datasets import load_dataset
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

Load Data

In [3]:
df = pd.read_parquet("hf://datasets/FronkonGames/steam-games-dataset/data/train-00000-of-00001.parquet")

In [4]:
df.sample(5)

,appID,name,release_date,estimated_owners,peak_ccu,required_age,price,dlc_count,detailed_description,short_description,...,median_playtime_forever,median_playtime_2weeks,developers,publishers,categories,genres,tags,screenshots,movies,packages
78521,579160,A God-Like Backhand!,"Feb 16, 2017",0 - 20000,0,0,0.99,0,A God vs. Ragdoll arcade style game.,A God vs. Ragdoll arcade style game.,...,0,0,[Q-Ball Games LLC],[Q-Ball Games LLC],"[Single-player, Tracked Controller Support, VR...","[Action, Adventure, Casual, Indie, Simulation]",[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy A God-Like Backhand!"", ""descri..."
20532,4104260,Tidy Up,"Nov 21, 2025",0 - 20000,0,0,1.99,0,Tidy Up: The Inventory Sorting Game This is a ...,Tidy Up is a cozy inventory-sorting game where...,...,0,0,[GameShock],[GameShock],"[Single-player, Custom Volume Controls, Keyboa...","[Casual, Indie]",[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Tidy Up"", ""description"": """", ""..."
14017,3994630,Near The Fear Playtest,"Sep 13, 2025",0 - 0,0,0,0.00,0,,,...,0,0,[],[],[],[],[],[],[],[]
111773,765400,Christmas Tale - Visual Novel,"Jan 2, 2018",0 - 20000,0,0,1.19,1,Christmas tale - it's a short choice-based vis...,Christmas tale - it's a short choice-based vis...,...,0,0,[morojenoe's empire],[morojenoe's empire],"[Single-player, Steam Achievements, Family Sha...","[Adventure, Casual]",[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Christmas Tale - Visual Novel""..."
76452,3116200,Poker Tower Defense,"Aug 6, 2024",0 - 0,0,0,0.59,0,This is a tower defense game played with playi...,This is a tower defense game played with playi...,...,0,0,[African boy Keita studio],[African boy Keita studio],"[Single-player, Family Sharing]","[Casual, Indie, Strategy]",[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Poker Tower Defense"", ""descrip..."


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 124146 entries, 0 to 124145
Data columns (total 41 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   appID                     124146 non-null  str    
 1   name                      124145 non-null  str    
 2   release_date              124146 non-null  str    
 3   estimated_owners          124146 non-null  str    
 4   peak_ccu                  124146 non-null  int64  
 5   required_age              124146 non-null  int64  
 6   price                     124146 non-null  float64
 7   dlc_count                 124146 non-null  int64  
 8   detailed_description      115718 non-null  str    
 9   short_description         115831 non-null  str    
 10  supported_languages       124146 non-null  object 
 11  full_audio_languages      124146 non-null  object 
 12  reviews                   12124 non-null   str    
 13  header_image              124065 non-null  str    
 14 

## Drop Columns

The initial `df.info()` check showed no missing values. However, many of the columns had missing data encode as empty strings or placeholders instead of actual NaN values

After analyzing each column, I decided to drop some of them:

*   Mostly '0': `user_score`, `recommendations`
*   Mostly blank: `reviews`, `metacritic_url`, `score_rank`, `notes`
*   No use for analysis: `detailed_description`, `short_description`, `header_image`, `website`, `support_url`, `support_email`, `screenshots`, `movies`

In [ ]:
df = df.replace(r'^\s*$', np.nan, regex=True)
df.isnull().sum()

appID                            0
name                             1
release_date                     0
estimated_owners                 0
peak_ccu                         0
required_age                     0
price                            0
dlc_count                        0
detailed_description          8428
short_description             8315
supported_languages              0
full_audio_languages             0
reviews                     112022
header_image                    81
website                      74081
support_url                  69397
support_email                22409
windows                          0
mac                              0
linux                            0
metacritic_score                 0
metacritic_url              119889
user_score                       0
positive                         0
negative                         0
score_rank                  124106
achievements                     0
recommendations                  0
notes               

In [6]:
df_clean = df.drop(columns=['user_score', 'score_rank', 'notes', 'reviews', 'metacritic_url', 'detailed_description', 'short_description', 'header_image', 'website', 'support_url', 'recommendations', 'screenshots', 'movies', 'support_email']).copy()

## Fix Data Type

There are several columns needed to be converted

In [ ]:
# String to Integer
df_clean['appID'] = df_clean['appID'].astype(int)
# Formatting Date
df_clean['release_date'] = pd.to_datetime(df_clean['release_date'])
# int64 -> int16 for faster processing and save memory
col_int16 = ['required_age', 
             'dlc_count', 
             'metacritic_score', 
             'achievements', 
             'average_playtime_2weeks', 
             'median_playtime_2weeks']
df_clean[col_int16] = df_clean[col_int16].astype('int16')

In [ ]:
# Save the cleaned dataset for analysis
df_clean.to_parquet('output.parquet', engine='pyarrow', compression='zstd')